# Selective ensemble: ARIMA/calendar + Chronos-2 + CatBoost

Чистый воспроизводимый пайплайн для `submission_catboost_chronos_selective.csv`.

- входная основа: `submission_arima_w30_route5_85k.csv`;
- Chronos-2 корректирует суточный уровень всех действующих маршрутов с весом 20%;
- CatBoost обучается только для маршрутов 11, 12, 25 и 28, где он улучшил временную валидацию;
- маршрут 5 сохраняется из основы на уровне 85k.


In [ ]:
import importlib.util
import subprocess
import sys

packages = []
if importlib.util.find_spec("chronos") is None:
    packages += ["chronos-forecasting==2.3.2", "pyarrow>=15"]
if importlib.util.find_spec("catboost") is None:
    packages += ["catboost==1.2.10"]
if packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])


In [ ]:
from pathlib import Path
import os
import random

os.environ.setdefault("HF_HOME", str(Path(".hf-cache").resolve()))

import numpy as np
import pandas as pd
import torch
from catboost import CatBoostRegressor
from chronos import Chronos2Pipeline

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

ROUTES = [1, 5, 7, 11, 12, 17, 25, 26, 28, 50]
ACTIVE_ROUTES = [route for route in ROUTES if route != 5]
SELECTED_ROUTES = [11, 12, 25, 28]
CATBOOST_WEIGHTS = {11: 0.35, 12: 0.10, 25: 0.35, 28: 0.45}
CHRONOS_WEIGHT = 0.20
CHRONOS_QUANTILE = "0.7"
KEYS = ["route", "date", "hour"]

CATBOOST_PARAMS = {
    11: dict(loss_function="Quantile:alpha=0.7", iterations=1500, learning_rate=0.09739630194717583, depth=4, l2_leaf_reg=0.005817066538204014, random_strength=0.011762426499887868, bagging_temperature=0.3954299769492494, border_count=239),
    12: dict(loss_function="MAE", iterations=1500, learning_rate=0.13944792632118025, depth=4, l2_leaf_reg=0.13444305532609163, random_strength=0.7017156693033213, bagging_temperature=0.8848219633796027, border_count=217),
    25: dict(loss_function="Quantile:alpha=0.7", iterations=1500, learning_rate=0.03428475736257873, depth=5, l2_leaf_reg=0.3119140149980465, random_strength=1.1005657369494315e-08, bagging_temperature=0.431920380647568, border_count=122),
    28: dict(loss_function="RMSE", iterations=1500, learning_rate=0.04835199705442417, depth=5, l2_leaf_reg=0.0034603811977855785, random_strength=8.767862485337334e-07, bagging_temperature=0.5720601179359943, border_count=153),
}
for params in CATBOOST_PARAMS.values():
    params.update(random_seed=SEED, thread_count=-1, verbose=False, allow_writing_files=False)

print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


In [ ]:
def first_existing(relative_paths, filename):
    for path in map(Path, relative_paths):
        if path.exists():
            return path
    kaggle_root = Path("/kaggle/input")
    if kaggle_root.exists():
        matches = sorted(kaggle_root.rglob(filename))
        if matches:
            return matches[0]
    raise FileNotFoundError(f"Не найден {filename}")

TRAIN_PATH = first_existing(["data/labels_day_train.csv", "data/labels/labels_day_train.csv"], "labels_day_train.csv")
HISTORY_TEST_PATH = first_existing(["data/labels_day_test.csv", "data/labels/labels_day_test.csv"], "labels_day_test.csv")
TEMPLATE_PATH = first_existing(["data/test_submission.csv"], "test_submission.csv")
BASELINE_PATH = first_existing(["submission_arima_w30_route5_85k.csv", "data/submission_arima_w30_route5_85k.csv"], "submission_arima_w30_route5_85k.csv")
OUTPUT_PATH = Path("/kaggle/working/submission_selective.csv") if Path("/kaggle/working").exists() else Path("submission_selective.csv")

history = pd.concat([pd.read_csv(TRAIN_PATH, sep=";"), pd.read_csv(HISTORY_TEST_PATH, sep=";")], ignore_index=True)
history["date"] = pd.to_datetime(history["date"])
template = pd.read_csv(TEMPLATE_PATH, sep=";")
template["date"] = pd.to_datetime(template["date"])
baseline = pd.read_csv(BASELINE_PATH, sep=";")
baseline["date"] = pd.to_datetime(baseline["date"])
assert template[KEYS].astype(str).equals(baseline[KEYS].astype(str))
print("Train:", TRAIN_PATH)
print("Baseline:", BASELINE_PATH)
print("Output:", OUTPUT_PATH)


In [ ]:
LEGACY_HOLIDAYS = set(pd.to_datetime([
    "2025-01-01", "2025-01-02", "2025-01-03", "2025-01-04",
    "2025-01-05", "2025-01-06", "2025-01-07", "2025-01-08",
    "2025-02-23", "2025-03-08", "2025-05-01", "2025-05-02",
    "2025-05-08", "2025-05-09", "2025-06-12", "2025-06-13",
    "2025-11-03", "2025-11-04", "2025-12-31",
]))

def seasonal_features(frame):
    frame = frame.copy()
    dt = frame["date"]
    frame["hour_val"] = frame["hour"]
    frame["dayofweek"] = dt.dt.dayofweek
    frame["day"] = dt.dt.day
    frame["month"] = dt.dt.month
    frame["dayofyear"] = dt.dt.dayofyear
    frame["weekofyear"] = dt.dt.isocalendar().week.astype(int)
    frame["quarter"] = dt.dt.quarter
    frame["is_weekend"] = frame["dayofweek"].isin([5, 6]).astype(int)
    frame["is_month_start"] = dt.dt.is_month_start.astype(int)
    frame["is_month_end"] = dt.dt.is_month_end.astype(int)
    frame["is_holiday"] = dt.isin(LEGACY_HOLIDAYS).astype(int)
    frame["is_off_day"] = ((frame["is_weekend"] == 1) | (frame["is_holiday"] == 1)).astype(int)
    frame["is_preholiday"] = (dt + pd.Timedelta(days=1)).isin(LEGACY_HOLIDAYS).astype(int)
    for name, period, value in [("hour", 24, frame["hour_val"]), ("dow", 7, frame["dayofweek"]), ("doy", 365.25, frame["dayofyear"]), ("month", 12, frame["month"])]:
        frame[f"{name}_sin"] = np.sin(2 * np.pi * value / period)
        frame[f"{name}_cos"] = np.cos(2 * np.pi * value / period)
    hour_in_week = frame["dayofweek"] * 24 + frame["hour_val"]
    for k in [1, 2, 3]:
        frame[f"fourier_week_sin_{k}"] = np.sin(2 * np.pi * k * hour_in_week / 168)
        frame[f"fourier_week_cos_{k}"] = np.cos(2 * np.pi * k * hour_in_week / 168)
    hour_in_year = (frame["dayofyear"] - 1) * 24 + frame["hour_val"]
    for k in [1, 2]:
        frame[f"fourier_year_sin_{k}"] = np.sin(2 * np.pi * k * hour_in_year / 8766)
        frame[f"fourier_year_cos_{k}"] = np.cos(2 * np.pi * k * hour_in_year / 8766)
    return frame

def add_profiles(frame, train_mask):
    result = frame.copy()
    source = result.loc[train_mask]
    specs = [(["route", "hour", "dayofweek"], "rhd", ["mean", "median", "std"]), (["route", "hour", "is_off_day"], "rho", ["mean", "median"]), (["route", "hour"], "rh", ["mean", "median"])]
    for keys, tag, stats in specs:
        profile = source.groupby(keys)["boardings"].agg(stats).reset_index()
        profile = profile.rename(columns={stat: f"profile_{tag}_{stat}" for stat in stats})
        result = result.merge(profile, on=keys, how="left", validate="many_to_one")
    return result

cat_grid = pd.MultiIndex.from_product([SELECTED_ROUTES, pd.date_range("2025-01-01", "2025-12-31"), range(24)], names=KEYS).to_frame(index=False)
cat_grid = cat_grid.merge(history[KEYS + ["boardings"]], on=KEYS, how="left", validate="one_to_one")
cat_grid["boardings"] = cat_grid["boardings"].fillna(0.0)
cat_grid = seasonal_features(cat_grid)
cat_train_mask = cat_grid["date"] <= "2025-10-31"
cat_grid = add_profiles(cat_grid, cat_train_mask)
CAT_FEATURES = [column for column in cat_grid.columns if column not in {"route", "date", "boardings"}]


In [ ]:
cat_predictions = []
for route in SELECTED_ROUTES:
    train_route = cat_grid.loc[cat_train_mask & cat_grid["route"].eq(route)]
    future_route = cat_grid.loc[cat_grid["date"].between("2025-11-01", "2025-12-31") & cat_grid["route"].eq(route)]
    model = CatBoostRegressor(**CATBOOST_PARAMS[route])
    model.fit(train_route[CAT_FEATURES], train_route["boardings"])
    prediction = np.maximum(0, model.predict(future_route[CAT_FEATURES]))
    part = future_route[KEYS].copy()
    part["catboost_prediction"] = prediction
    cat_predictions.append(part)
    print("CatBoost route", route, "done")
cat_predictions = pd.concat(cat_predictions, ignore_index=True)


In [ ]:
def russian_calendar_2025():
    dates = pd.date_range("2025-01-01", "2025-12-31", freq="D")
    weekends = set(dates[dates.dayofweek >= 5])
    extra_off = set(pd.to_datetime(["2025-01-01", "2025-01-02", "2025-01-03", "2025-01-04", "2025-01-05", "2025-01-06", "2025-01-07", "2025-01-08", "2025-05-01", "2025-05-02", "2025-05-08", "2025-05-09", "2025-06-12", "2025-06-13", "2025-11-03", "2025-11-04", "2025-12-31"]))
    off_days = (weekends | extra_off) - {pd.Timestamp("2025-11-01")}
    preholidays = set(pd.to_datetime(["2025-03-07", "2025-04-30", "2025-06-11", "2025-11-01"]))
    return off_days, preholidays

OFF_DAYS, PREHOLIDAYS = russian_calendar_2025()
def calendar_frame(routes, dates):
    frame = pd.MultiIndex.from_product([routes, pd.DatetimeIndex(dates)], names=["item_id", "timestamp"]).to_frame(index=False)
    dt = frame["timestamp"]
    dow, doy = dt.dt.dayofweek, dt.dt.dayofyear
    frame["dow_sin"] = np.sin(2 * np.pi * dow / 7)
    frame["dow_cos"] = np.cos(2 * np.pi * dow / 7)
    frame["doy_sin"] = np.sin(2 * np.pi * doy / 365.25)
    frame["doy_cos"] = np.cos(2 * np.pi * doy / 365.25)
    frame["is_off_day"] = dt.isin(OFF_DAYS).astype(float)
    frame["is_preholiday"] = dt.isin(PREHOLIDAYS).astype(float)
    frame["is_month_start"] = dt.dt.is_month_start.astype(float)
    frame["is_month_end"] = dt.dt.is_month_end.astype(float)
    return frame

history_dates = pd.date_range("2025-01-01", "2025-10-31")
future_dates = pd.date_range("2025-11-01", "2025-12-31")
daily_observed = history.groupby(["route", "date"], as_index=False)["boardings"].sum()
daily = pd.MultiIndex.from_product([ACTIVE_ROUTES, history_dates], names=["route", "date"]).to_frame(index=False)
daily = daily.merge(daily_observed, on=["route", "date"], how="left")
daily["boardings"] = daily["boardings"].fillna(0.0)
context_df = calendar_frame(ACTIVE_ROUTES, history_dates).merge(daily, left_on=["item_id", "timestamp"], right_on=["route", "date"], validate="one_to_one").drop(columns=["route", "date"]).rename(columns={"boardings": "target"})
future_df = calendar_frame(ACTIVE_ROUTES, future_dates)


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
pipeline = Chronos2Pipeline.from_pretrained("amazon/chronos-2", device_map=device, dtype=dtype)
chronos_daily = pipeline.predict_df(context_df, future_df=future_df, prediction_length=61, quantile_levels=[0.7], batch_size=100, context_length=512, cross_learning=True, freq="D")
chronos_daily = chronos_daily.rename(columns={"item_id": "route", "timestamp": "date"})
chronos_daily[CHRONOS_QUANTILE] = chronos_daily[CHRONOS_QUANTILE].clip(lower=0)
print("Chronos rows:", len(chronos_daily))


In [ ]:
active = baseline.loc[baseline["route"] != 5].copy().rename(columns={"prediction": "baseline"})
daily_baseline = active.groupby(["route", "date"])["baseline"].transform("sum")
active["hour_share"] = np.divide(active["baseline"], daily_baseline, out=np.full(len(active), 1 / 24, dtype=float), where=daily_baseline.to_numpy() > 0)
active = active.merge(chronos_daily[["route", "date", CHRONOS_QUANTILE]], on=["route", "date"], validate="many_to_one")
active["chronos_hourly"] = active["hour_share"] * active[CHRONOS_QUANTILE]
active["chronos_blend"] = (1 - CHRONOS_WEIGHT) * active["baseline"] + CHRONOS_WEIGHT * active["chronos_hourly"]
active = active.merge(cat_predictions, on=KEYS, how="left", validate="one_to_one")
active["catboost_weight"] = active["route"].map(CATBOOST_WEIGHTS).fillna(0.0)
active["prediction"] = (1 - active["catboost_weight"]) * active["chronos_blend"] + active["catboost_weight"] * active["catboost_prediction"].fillna(0.0)

output = baseline.copy()
output["prediction"] = output["prediction"].astype(float)
lookup = active.set_index(KEYS)["prediction"]
mask = output["route"] != 5
output.loc[mask, "prediction"] = pd.MultiIndex.from_frame(output.loc[mask, KEYS]).map(lookup)
output["prediction"] = np.rint(output["prediction"].clip(lower=0)).astype(int)
output["date"] = output["date"].dt.strftime("%Y-%m-%d")
assert len(output) == 14_640
assert output.isna().sum().sum() == 0
assert output.duplicated(KEYS).sum() == 0
assert output.loc[output["route"] == 5, "prediction"].sum() in range(84_990, 85_011)
assert template[KEYS].astype(str).equals(output[KEYS].astype(str))
output.to_csv(OUTPUT_PATH, sep=";", index=False)
print("Saved:", OUTPUT_PATH.resolve())
print("Rows:", len(output), "Total:", output["prediction"].sum())
reference_path = Path("submission_catboost_chronos_selective.csv")
if reference_path.exists():
    reference = pd.read_csv(reference_path, sep=";")
    difference = (output["prediction"] - reference["prediction"]).abs()
    assert difference.max() <= 1, "Воспроизведённый прогноз заметно отличается от reference"
    print("Reference within ±1:", bool((difference <= 1).all()), "max diff:", difference.max())
display(output.head())
